# Granite-type classification and repeated OOF attribution audit (Task B v6)

This notebook retains the v5 I/A/S workflow that produces the class-specific SHAP violin figure. It adds only the confirmatory evidence needed before Part 4: repeated source-connected nested OOF performance, nested probability calibration, repeated class-specific OOF SHAP, source-block stability and a six-feature bridge contract.

Task A and Part 4 are not read. They cannot influence labels, algorithm selection, tuning or readiness thresholds. The fixed held-out partition remains a one-time check and is not reused as the v6 readiness gate.

Run cells in order. Model comparison and repeated nested validation are computationally intensive. Persistent Optuna studies allow the same formal run to resume after interruption.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

def locate_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_path = candidate / 'config' / 'granite_classification_config.json'
        if config_path.exists():
            return candidate.resolve()
    raise FileNotFoundError('Could not locate the formal Task B configuration.')

PROJECT_ROOT = locate_project_root()
SOURCE_DIR = PROJECT_ROOT / 'src'
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

CONFIG_PATH = PROJECT_ROOT / 'config' / 'granite_classification_config.json'
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))

from classification_pipeline import (
    compare_models, final_evaluation, generate_shap, output_paths, prepare_data,
    validate_final_outputs, validate_inputs, write_manifest,
)
from data_audit import run_all_audit
from robustness_audit import run_robustness_audit

print('Project root:', PROJECT_ROOT)
print('Analysis revision:', config['analysis_revision'])
print('Output root:', config['paths']['output_root'])
print('Primary bridge:', config['bridge_features']['primary_reproduced_in_task_a'])
print('Legacy sensitivity:', config['bridge_features']['legacy_sensitivity_only'])

## 1. Preflight audit

This cell reads data and validates class/source-block coverage only. It does not fit a model.

In [ ]:
preflight = validate_inputs(config)
print(json.dumps(preflight, ensure_ascii=False, indent=2))

## 2. Frozen v5 modelling core

The next cell prepares the source-connected holdout, compares RF/SVM/MLP/XGBoost on development nested OOF, evaluates the fixed held-out partition once, and generates the original canonical OOF TreeSHAP and SHAP figures.

In [ ]:
paths = output_paths(config)
preparation = prepare_data(config)
selection_path = paths['comparison'] / 'model_selection.json'
final_parameters_path = paths['final'] / 'selected_model_hyperparameters.json'
holdout_contract_path = paths['processed'] / 'holdout_selection_contract.json'
core_complete = selection_path.exists() and final_parameters_path.exists() and holdout_contract_path.exists()
if core_complete:
    holdout_state = json.loads(holdout_contract_path.read_text(encoding='utf-8'))
    core_complete = bool(holdout_state.get('final_evaluation_completed', False))

if core_complete:
    selection_payload = json.loads(selection_path.read_text(encoding='utf-8'))
    comparison = {'selection': selection_payload}
    final_result = {'status': 'existing one-time held-out result retained'}
    print('Resume mode: model comparison and one-time held-out evaluation already complete; skipping both.')
else:
    comparison = compare_models(config)
    final_result = final_evaluation(config)
canonical_shap = generate_shap(config)

print('Raw S1 records:', preparation['summary']['raw_S1_n'])
print('Explicit I/A/S records:', preparation['summary']['explicit_IAS_n'])
print('Post-filter analysis cohort:', preparation['summary']['post_protocol_filter_n'])
print('Development records:', preparation['summary']['development_n'])
print('Fixed held-out records:', preparation['summary']['heldout_n'])
print('Selected algorithm:', comparison['selection']['best_ranked_model'])
print('Interpretation model:', comparison['selection'].get('interpretation_model'))
print('Canonical valid OOF SHAP records:', canonical_shap.get('valid_oof'))

## 3. Original data/domain and SHAP-violin audit

This retains the old-version data audit, primary-domain summary and class-specific SHAP violin output. Flags do not delete or relabel records.

In [ ]:
audit_result = run_all_audit(PROJECT_ROOT, config)
print('Data-audit rows:', audit_result['data_audit_rows'])
print('Flag counts:', audit_result['data_audit_flags'])
print('\nPrimary-domain metrics:')
print(pd.DataFrame(audit_result['primary_domain_metrics']).to_string(index=False))

## 4. Confirmatory repeated validation and SHAP stability

This stage does not compare algorithms again. It retunes the already selected algorithm inside each development outer-training partition, fits calibration from inner OOF predictions, and then repeats full-data OOF TreeSHAP for attribution stability. Part 4 is still not read.

In [ ]:
robustness = run_robustness_audit(config)
readiness = robustness['readiness']
print('Repeated development audit:', robustness['repeated_development_validation'])
print('Repeated SHAP audit:', robustness['repeated_full_data_shap'])
print('Readiness level:', readiness['readiness_level'])
print('Statistical robustness gate:', readiness['statistical_robustness_gate_passed'])
print('Label-source independence verified:', readiness['label_source_independence_verified'])

## 5. Reproducibility manifest and final contract check

The manifest is written only after all core and robustness outputs exist.

In [ ]:
manifest = write_manifest(config, CONFIG_PATH)
acceptance = validate_final_outputs(config)
print(json.dumps(acceptance, ensure_ascii=False, indent=2))
print('Manifest outputs:', len(manifest['outputs']))

## Interpretation boundary

- Use exploratory_ready for an exploratory cross-task bridge; use internally_robust_for_linkage only when all statistical gates and the manual label-source checkpoint pass.
- The six-feature canonical bridge is Rb–CaO–Nb–Zr–P2O5–Ba. MgO–Na2O–K2O remain sensitivity only.
- SHAP correspondence supports model-derived geochemical consistency, not causal metallogenic proof.
- After any Task B probability or SHAP change, Part 4 must rebuild the Record ID joint cohort and recompute every coupling statistic and figure.